# Notebook 1: Data Pipeline & Preprocessing (ETL)

Notebook ini berfokus murni pada fase **Extract, Transform, Load (ETL)** tanpa mengeksekusi pelatihan Machine Learning berat.

**Aktivitas Utama:**
1. Membaca dataset lokal `IDSMSA.csv` (Twitter/X) dan `Dataset-CNBCI-Sentimented.csv` (Berita CNBC).
2. Menjalankan fungsi pembersihan teks (hapus URL, mention, tagar, standarisasi).
3. Menstandardisasi label kelas menjadi `['Positif', 'Netral', 'Negatif']`.
4. Menggabungkan semuanya menjadi satu dataframe yang diacak (*shuffled*).
5. Menyimpan dataframe akhir ke folder lokal (`dataset_gabungan_final.csv`).

*Catatan: Anda tidak memerlukan GPU (cukup CPU) untuk menjalankan notebook ini.*

### Import Library

In [ ]:
# Sel 1: Import Library Utama
import pandas as pd
import re
import string
import os

print("[INFO] Library berhasil dimuat.")

### Membaca Dataset
Mengunggah dan membaca file `IDSMSA.csv` dan `Dataset-CNBCI-Sentimented.csv`. 
*Pastikan Anda sudah meletakkan file tersebut di direktori yang sama dengan notebook ini (atau unggah ke Colab).* 

In [ ]:
# Sel 2: Membaca Dataset

# 1. Load ID-SMSA (Twitter Scraping)
# Asumsi nama file: IDSMSA.csv
print("[ETL] Membaca dataset ID-SMSA...")
try:
    df_idsmsa = pd.read_csv('IDSMSA.csv')
    print(f"  -> Berhasil! Total baris: {len(df_idsmsa)}")
    # Pilih kolom yang relevan: Sentence (teks), Sentiment (label)
    df_idsmsa = df_idsmsa[['Sentence', 'Sentiment']].copy()
    # Seragamkan nama kolom
    df_idsmsa.rename(columns={'Sentence': 'text', 'Sentiment': 'label_raw'}, inplace=True)
except FileNotFoundError:
    print("  [ERROR] File 'IDSMSA.csv' tidak ditemukan.")
    
# 2. Load Dataset CNBCI
# Asumsi nama file: Dataset-CNBCI-Sentimented.csv
print("\n[ETL] Membaca dataset CNBCI...")
try:
    df_cnbc = pd.read_csv('Dataset-CNBCI-Sentimented.csv')
    print(f"  -> Berhasil! Total baris: {len(df_cnbc)}")
    # Pilih kolom yang relevan: judul (teks), sentimen (label)
    df_cnbc = df_cnbc[['judul', 'sentimen']].copy()
    # Seragamkan nama kolom
    df_cnbc.rename(columns={'judul': 'text', 'sentimen': 'label_raw'}, inplace=True)
except FileNotFoundError:
    print("  [ERROR] File 'Dataset-CNBCI-Sentimented.csv' tidak ditemukan.")


### Pembersihan Teks (Cleaning) & Standarisasi Label
Fungsi ini menghapus kotoran dari hasil *scraping* (seperti `[USERNAME]`, `[URL]`, karakter spesial) dan mengubah semua ragam label menjadi tepat `['Positif', 'Netral', 'Negatif']`.

In [ ]:
# Sel 3: Fungsi Cleaning & Standarisasi

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
        
    text = text.lower()
    
    # Menghapus tag bawaan ID-SMSA seperti [USERNAME], [URL], [HASHTAG]
    text = re.sub(r'\[username\]|\[url\]|\[hashtag\]', '', text)
    
    # Menghapus URL biasa (jika masih ada)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Menghapus tag HTML
    text = re.sub(r'<.*?>', '', text)
    
    # Menghapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Menghapus spasi ganda
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def standardize_label(label: str) -> str:
    """
    Mengonversi label dari berbagai sumber (misal: 'Neutral', 'negatif') 
    menjadi standar baku: 'Positif', 'Netral', 'Negatif'.
    """
    if not isinstance(label, str):
        return "Netral"  # Default fallback
        
    l = label.lower().strip()
    
    if l in ['positive', 'positif', 'pos']:
        return 'Positif'
    elif l in ['negative', 'negatif', 'neg']:
        return 'Negatif'
    else:
        return 'Netral'


### Eksekusi Transformasi Data
Terapkan pembersihan dan standarisasi ke dalam masing-masing dataframe.

In [ ]:
# Sel 4: Eksekusi Transformasi

print("[ETL] Memproses Dataset ID-SMSA...")
if 'df_idsmsa' in locals():
    df_idsmsa['text_clean'] = df_idsmsa['text'].apply(clean_text)
    df_idsmsa['label'] = df_idsmsa['label_raw'].apply(standardize_label)
    df_idsmsa = df_idsmsa[['text_clean', 'label']].rename(columns={'text_clean': 'text'})

print("[ETL] Memproses Dataset CNBCI...")
if 'df_cnbc' in locals():
    df_cnbc['text_clean'] = df_cnbc['text'].apply(clean_text)
    df_cnbc['label'] = df_cnbc['label_raw'].apply(standardize_label)
    df_cnbc = df_cnbc[['text_clean', 'label']].rename(columns={'text_clean': 'text'})

print("[INFO] Transformasi Selesai.")

### Penggabungan dan Pengacakan (Shuffle)
Menggabungkan semuanya menjadi satu dataframe tunggal agar sebaran data representatif.

In [ ]:
# Sel 5: Menggabungkan (Concat) & Acak Data (Shuffle)

dfs_to_concat = []
if 'df_idsmsa' in locals(): dfs_to_concat.append(df_idsmsa)
if 'df_cnbc' in locals(): dfs_to_concat.append(df_cnbc)

if dfs_to_concat:
    # Gabungkan
    df_gabungan = pd.concat(dfs_to_concat, ignore_index=True)
    
    # Hapus duplikat teks (jika ada)
    df_gabungan.drop_duplicates(subset=['text'], inplace=True)
    
    # Hapus teks kosong
    df_gabungan = df_gabungan[df_gabungan['text'].str.strip() != '']
    
    # Acak baris secara acak (shuffle)
    df_gabungan = df_gabungan.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("\n[ETL] Dataset berhasil digabung!")
    print(f"Total baris unik yang akan disimpan: {len(df_gabungan)}")
    print("\nDistribusi Label Final:")
    print(df_gabungan['label'].value_counts())
else:
    print("[ERROR] Tidak ada dataframe yang berhasil dimuat.")

### Export Data Akhir
Menyimpan dataframe akhir (`dataset_gabungan_final.csv`) ke direktori lokal (root folder).

In [ ]:
# Sel 6: Simpan Data secara Lokal

if 'df_gabungan' in locals():
    output_file = 'dataset_gabungan_final.csv'
    
    print(f"[ETL] Sedang menyimpan dataset ke lokal: {output_file} ...")
    try:
        df_gabungan.to_csv(output_file, index=False, encoding='utf-8')
        print("\n✅ [SUKSES] Dataset berhasil disimpan!")
        print("Anda kini dapat beralih ke Notebook 2 (Training).")
    except Exception as e:
        print("\n❌ [GAGAL] Terjadi kesalahan saat menyimpan file.")
        print("Error detail:", str(e))
